In [1]:
import subprocess
print(subprocess.run(["find","/kaggle/input","-maxdepth","4","-type","d"],
                     capture_output=True,text=True).stdout)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/qianlanzz
/kaggle/input/datasets/qianlanzz/xbd-dataset
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd



In [2]:
import subprocess
print(subprocess.run(
    ["find", "/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd", "-maxdepth", "3"],
    capture_output=True, text=True).stdout[:4000])

/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/santa-rosa-wildfire_00000138_pre_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/hurricane-harvey_00000041_pre_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/hurricane-matthew_00000295_post_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/socal-fire_00000723_post_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/hurricane-michael_00000020_pre_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/midwest-flooding_00000033_post_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/socal-fire_00000385_post_disaster.json
/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1/labels/socal-fire_00001325_pre_disaster.

In [3]:
import subprocess, os
root = "/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd"
print("splits:", os.listdir(root))
print("tier1:", os.listdir(f"{root}/tier1"))

splits: ['tier1', 'tier3', 'test', 'hold', 'train']
tier1: ['labels', 'images', 'masks']


In [4]:
%%writefile /kaggle/working/01_prepare_damage.py

import argparse
import io
import json
import multiprocessing as mp
from collections import Counter, defaultdict
from pathlib import Path

import h5py
import numpy as np
from PIL import Image
from shapely import wkt as shapely_wkt

CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

PATCH_SIZE = 128 
CONTEXT_PAD = 10
MIN_AREA = 12


# ==================================================== Pass 1: read JSON only

def scan_one_json(json_path):
    jp = Path(json_path)
    base = jp.stem.replace("_post_disaster", "")
    event = base.rsplit("_", 1)[0]

    try:
        with open(jp) as f:
            meta = json.load(f)
    except Exception:
        return base, event, []

    feats = meta.get("features", {}).get("xy", [])
    out = []
    for i, feat in enumerate(feats):
        subtype = feat.get("properties", {}).get("subtype")
        if subtype not in CLASS_TO_IDX:
            continue
        try:
            poly = shapely_wkt.loads(feat["wkt"])
            if poly.area < MIN_AREA:
                continue
            minx, miny, maxx, maxy = poly.bounds
        except Exception:
            continue

        cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
        half = max(maxx - minx, maxy - miny) / 2 + CONTEXT_PAD
        box = (int(cx - half), int(cy - half), int(cx + half), int(cy + half))
        out.append((i, CLASS_TO_IDX[subtype], box))

    return base, event, out


def extract_one_scene(task):
    img_path, base, wanted = task
    if not wanted:
        return []

    try:
        img = Image.open(img_path).convert("RGB")
    except Exception:
        return []
    W, H = img.size

    results = []
    for poly_idx, label, (l, t, r, b) in wanted:
        l, t = max(0, l), max(0, t)
        r, b = min(W, r), min(H, b)
        if r - l < 4 or b - t < 4:
            continue
        patch = img.crop((l, t, r, b)).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)

        buf = io.BytesIO()
        patch.save(buf, format="PNG", optimize=False) 
        results.append((f"{base}_{poly_idx:04d}", label, buf.getvalue()))

    return results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--xbd_root", required=True)
    ap.add_argument("--splits", nargs="+", default=["train", "tier3"])
    ap.add_argument("--out", default="data/damage.h5")
    ap.add_argument("--workers", type=int, default=max(1, mp.cpu_count() - 1))
    ap.add_argument("--val_ratio", type=float, default=0.2)
    ap.add_argument("--cap_no_damage", type=int, default=100000,
                    help="max no-damage patches in train (0 = all)")
    ap.add_argument("--cap_other", type=int, default=0,
                    help="cap for the other classes (0 = keep all, usually correct)")
    ap.add_argument("--limit_scenes", type=int, default=0)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    rng = np.random.RandomState(args.seed)
    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    json_files = []
    for sp in args.splits:
        d = Path(args.xbd_root) / sp / "labels"
        if d.exists():
            json_files.extend(sorted(d.glob("*_post_disaster.json")))
        else:
            print(f"  (warning: {d} not found, skipping)")
    if args.limit_scenes:
        json_files = json_files[: args.limit_scenes]
    print(f"Pass 1: scanning {len(json_files)} JSON files ({args.workers} workers)...")

    with mp.Pool(args.workers) as pool:
        scanned = []
        for n, res in enumerate(pool.imap_unordered(scan_one_json,
                                                    [str(p) for p in json_files],
                                                    chunksize=16), 1):
            scanned.append(res)
            if n % 1000 == 0:
                print(f"  {n}/{len(json_files)}")

    total_polys = sum(len(x[2]) for x in scanned)
    print(f"Pass 1 done: {total_polys:,} valid building polygons found")

    dist = Counter()
    for _, _, polys in scanned:
        for _, lab, _ in polys:
            dist[lab] += 1
    print("\nOriginal distribution:")
    for i, c in enumerate(CLASSES):
        print(f"  {c:15s}: {dist[i]:8,d}  ({100*dist[i]/max(1,total_polys):5.2f}%)")

    by_event = defaultdict(list)
    for base, event, _ in scanned:
        by_event[event].append(base)

    val_scenes = set()
    for event, scenes in by_event.items():
        scenes = sorted(scenes)
        rng.shuffle(scenes)
        n_val = max(1, int(round(len(scenes) * args.val_ratio)))
        val_scenes.update(scenes[:n_val])
    print(f"\nSplit: {len(by_event)} events | val scenes {len(val_scenes)} / {len(scanned)}")

    caps = {i: (args.cap_other if args.cap_other else None) for i in range(len(CLASSES))}
    caps[0] = args.cap_no_damage if args.cap_no_damage else None

    train_pool = defaultdict(list)     # label -> [(base, poly_idx, box)]
    val_items = []
    for base, event, polys in scanned:
        is_val = base in val_scenes
        for poly_idx, lab, box in polys:
            if is_val:
                val_items.append((base, poly_idx, lab, box))
            else:
                train_pool[lab].append((base, poly_idx, box))

    train_items = []
    print("\nApplying cap to train:")
    for lab in range(len(CLASSES)):
        pool_l = train_pool[lab]
        cap = caps[lab]
        if cap and len(pool_l) > cap:
            idx = rng.choice(len(pool_l), size=cap, replace=False)
            chosen = [pool_l[i] for i in idx]
            print(f"  {CLASSES[lab]:15s}: {len(pool_l):8,d} -> {cap:8,d}")
        else:
            chosen = pool_l
            print(f"  {CLASSES[lab]:15s}: {len(pool_l):8,d} (all kept)")
        train_items.extend([(b, p, lab, bx) for b, p, bx in chosen])

    print(f"\nFinal: train {len(train_items):,} | val {len(val_items):,}")

    images_dirs = [Path(args.xbd_root) / sp / "images" for sp in args.splits]

    def find_image(base):
        for d in images_dirs:
            p = d / f"{base}_post_disaster.png"
            if p.exists():
                return p
        return None

    for split_name, items in [("train", train_items), ("val", val_items)]:
        grouped = defaultdict(list)
        for base, poly_idx, lab, box in items:
            grouped[base].append((poly_idx, lab, box))

        tasks = []
        for base, wanted in grouped.items():
            ip = find_image(base)
            if ip is not None:
                tasks.append((str(ip), base, wanted))

        print(f"\nPass 2 [{split_name}]: cutting {len(items):,} patches "
              f"from {len(tasks)} scenes...")

        # write to HDF5 as variable-length uint8 (PNG bytes)
        mode = "a" if out_path.exists() else "w"
        with h5py.File(out_path, mode) as h5:
            if split_name in h5:
                del h5[split_name]
            grp = h5.create_group(split_name)
            vlen = h5py.vlen_dtype(np.uint8)
            d_img = grp.create_dataset("png", (0,), maxshape=(None,), dtype=vlen)
            d_lab = grp.create_dataset("label", (0,), maxshape=(None,), dtype="i1")
            d_uid = grp.create_dataset(
                "uid", (0,), maxshape=(None,), dtype=h5py.string_dtype())

            written = 0
            with mp.Pool(args.workers) as pool:
                for n, res in enumerate(pool.imap_unordered(extract_one_scene, tasks,
                                                            chunksize=4), 1):
                    if not res:
                        continue
                    k = len(res)
                    d_img.resize((written + k,))
                    d_lab.resize((written + k,))
                    d_uid.resize((written + k,))
                    for j, (uid, lab, png) in enumerate(res):
                        d_img[written + j] = np.frombuffer(png, dtype=np.uint8)
                        d_lab[written + j] = lab
                        d_uid[written + j] = uid
                    written += k

                    if n % 200 == 0:
                        print(f"  {n}/{len(tasks)} scenes | {written:,} patches written")

            grp.attrs["classes"] = json.dumps(CLASSES)
            grp.attrs["patch_size"] = PATCH_SIZE
            print(f"  [{split_name}] {written:,} patches saved in total")

    size_gb = out_path.stat().st_size / 1e9
    print(f"\nDone -> {out_path}  ({size_gb:.2f} GB)")


if __name__ == "__main__":
    mp.set_start_method("fork", force=True) 
    main()

Writing /kaggle/working/01_prepare_damage.py


In [5]:
!python /kaggle/working/01_prepare_damage.py \
    --xbd_root /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd \
    --splits tier1 tier3 train \
    --out /kaggle/working/damage.h5 \
    --workers 4 \
    --cap_no_damage 100000

  (warning: /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/train/labels not found, skipping)
Pass 1: scanning 9168 JSON files (4 workers)...
  1000/9168
  2000/9168
  3000/9168
  4000/9168
  5000/9168
  6000/9168
  7000/9168
  8000/9168
  9000/9168
Pass 1 done: 304,370 valid building polygons found

Original distribution:
  no-damage      :  233,635  (76.76%)
  minor-damage   :   25,724  ( 8.45%)
  major-damage   :   21,449  ( 7.05%)
  destroyed      :   23,562  ( 7.74%)

Split: 19 events | val scenes 1837 / 9168

Applying cap to train:
  no-damage      :  188,564 ->  100,000
  minor-damage   :   19,997 (all kept)
  major-damage   :   17,083 (all kept)
  destroyed      :   18,685 (all kept)

Final: train 155,765 | val 60,041

Pass 2 [train]: cutting 155,765 patches from 4257 scenes...
  200/4257 scenes | 24,169 patches written
  400/4257 scenes | 42,795 patches written
  600/4257 scenes | 61,203 patches written
  800/4257 scenes | 73,873 patches written
  1000/4257 scenes | 84,993 pa